# FinChart-R2 — Phase 2C Teacher Raw Capture (Local)

This notebook runs locally against `FinChart-R2/.env` and captures raw teacher messages for later analysis. It reads only ChartQA train-mining errors, saves no credentials, and does not create DPO pairs.


## 1. Install local dependencies

Run this once in the notebook kernel. The API configuration remains in `.env`; do not paste it into the notebook.


In [ ]:
# %pip -q install -U "httpx<1.0" datasets requests pillow python-dotenv
# print('Dependencies ready. Restart the notebook kernel once after this cell, then run from Cell 2.')


## 2. Local project configuration

The paths below must point to your existing FinChart-R2 project. This cell only checks that `.env` exists; it never displays its contents.


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path(r'D:\Finance_AI\FinChart-R2')
INPUT_JSONL = PROJECT_ROOT / 'results' / 'finchart_r2_phase2c_train_mining' / 'dpo_train_error.jsonl'
OUTPUT_DIR = PROJECT_ROOT / 'results' / 'finchart_r2_phase2c_train_mining'
SCRIPT_PATH = PROJECT_ROOT / 'scripts' / 'capture_phase2c_teacher_raw.py'

AUDIT_LIMIT = 906
STRUCTURED_MODE = 'json_object'
RETRY_FAILED = False

assert PROJECT_ROOT.is_dir(), f'Project not found: {PROJECT_ROOT}'
assert (PROJECT_ROOT / '.env').is_file(), 'Create FinChart-R2/.env before running; its contents are not printed.'
assert INPUT_JSONL.is_file(), f'Input not found: {INPUT_JSONL}'
assert SCRIPT_PATH.is_file(), f'Raw-capture script not found: {SCRIPT_PATH}'
assert STRUCTURED_MODE in {'json_schema', 'json_object'}
print('Input:', INPUT_JSONL)
print('Output:', OUTPUT_DIR)
print('Teacher configuration: loaded by the script from .env (values hidden).')


## 3. Optional: inspect one request before batch capture

Set `RUN_DEBUG_SAMPLE = True` to make one paid API request. It displays the chart, question, reference answer, raw SFT response, teacher prompt, and raw teacher message—but never the API key.


In [ ]:
RUN_DEBUG_SAMPLE = True
DEBUG_SAMPLE_OFFSET = 0

if RUN_DEBUG_SAMPLE:
    import importlib.util
    import json
    import requests
    from datasets import load_dataset
    from IPython.display import display

    records = [json.loads(line) for line in INPUT_JSONL.read_text(encoding='utf-8').splitlines() if line.strip()]
    row = records[DEBUG_SAMPLE_OFFSET]
    dataset = load_dataset('HuggingFaceM4/ChartQA', split=row['image_split'])
    example = dataset[int(row['image_index'])]
    assert str(example.get('query', example.get('question', ''))).strip() == str(row['question']).strip()

    spec = importlib.util.spec_from_file_location('phase2c_raw_capture', SCRIPT_PATH)
    capture = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(capture)
    config = capture.teacher_config()  # reads .env without printing its values
    prompt = capture.prompt_for(row)

    print('=== CHART ===')
    display(example['image'])
    print('=== HUMAN-READABLE INPUT ===')
    print(json.dumps({key: row.get(key) for key in ['dataset_index', 'image_split', 'image_index', 'question', 'ground_truth', 'sft_prediction_raw', 'extracted_final_answer']}, ensure_ascii=False, indent=2))
    print('=== TEACHER PROMPT ===')
    print(prompt)

    url = config['base_url'].rstrip('/')
    if not url.endswith('chat/completions'):
        url += '/chat/completions'
    response_format = {'type': 'json_schema', 'json_schema': {'name': 'phase2c_raw_capture', 'strict': False, 'schema': capture.MINIMAL_SCHEMA}} if STRUCTURED_MODE == 'json_schema' else {'type': 'json_object'}
    payload = {
        'model': config['model'], 'temperature': 0, 'max_tokens': 700, 'response_format': response_format,
        'messages': [
            {'role': 'system', 'content': 'Return only the compact JSON object requested by the user.'},
            {'role': 'user', 'content': [
                {'type': 'text', 'text': prompt},
                {'type': 'image_url', 'image_url': {'url': 'data:image/png;base64,' + capture.image_as_base64(example['image'])}},
            ]},
        ],
    }
    response = requests.post(url, headers={'Authorization': f"Bearer {config['key']}", 'Content-Type': 'application/json'}, json=payload, timeout=120)
    print('=== HTTP STATUS ===', response.status_code)
    response.raise_for_status()
    message = response.json()['choices'][0]['message']
    print('=== RAW TEACHER MESSAGE ===')
    print(json.dumps(message, ensure_ascii=False, indent=2))
    try:
        print('=== PARSED JSON ===')
        print(json.dumps(capture.parse_json_response(message.get('content')), ensure_ascii=False, indent=2))
    except Exception as exc:
        print('=== PARSE ERROR ===')
        print(f'{type(exc).__name__}: {exc}')
else:
    print('Debug disabled. Set RUN_DEBUG_SAMPLE = True to inspect one paid teacher request.')


## 4. Run resumable raw teacher capture

Start with 10. The output is `dpo_train_teacher_raw_capture.jsonl`; it preserves raw content and separate reasoning fields for later analysis. Existing successful captures are skipped. Set `RETRY_FAILED = True` only to retry API-error or unparseable rows.


In [ ]:
import subprocess
import sys

command = [
    sys.executable, str(SCRIPT_PATH),
    '--input', str(INPUT_JSONL),
    '--output-dir', str(OUTPUT_DIR),
    '--limit', str(AUDIT_LIMIT),
    '--workers', '1',
    '--structured-mode', STRUCTURED_MODE,
]
if RETRY_FAILED:
    command.append('--retry-failed')
print('Running:', ' '.join(command))
completed = subprocess.run(command, text=True, capture_output=True)
if completed.stdout:
    print('=== CAPTURE STDOUT ===')
    print(completed.stdout)
if completed.stderr:
    print('=== CAPTURE STDERR ===')
    print(completed.stderr)
if completed.returncode != 0:
    raise RuntimeError(f'Raw-capture script failed with exit code {completed.returncode}; see STDERR above.')


## 5. Inspect raw-capture result

This report is for collection quality only. Do not train DPO until a later analysis has selected schema-safe, ground-truth-consistent preference pairs.


In [ ]:
import json

report_path = OUTPUT_DIR / 'dpo_train_teacher_raw_capture_report.json'
capture_path = OUTPUT_DIR / 'dpo_train_teacher_raw_capture.jsonl'
report = json.loads(report_path.read_text(encoding='utf-8'))
count = sum(1 for line in capture_path.open(encoding='utf-8') if line.strip())
print(json.dumps(report, indent=2))
print('Raw captured records:', count)
print('Raw capture file:', capture_path)
